# AgriSense Cloud - train the two image models
Runtime > Change runtime type > **T4 GPU**, then run the cells in order. At the end `agrisense_image_models.zip` downloads. Unzip it into the project's `models` folder and run `python merge_image_meta.py`. The printed validation accuracies are your real results.

In [ ]:
!pip install -q kagglehub tifffile

## 1. Leaf-disease model (PlantVillage, Kaggle copy)

In [ ]:
import kagglehub, os, tensorflow as tf
SIZE = 128
root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
data_dir = next(dp for dp, dn, fn in os.walk(root) if os.path.basename(dp) == "color")
print("Using:", data_dir)
tr = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="training", seed=42, image_size=(SIZE, SIZE), batch_size=32)
va = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="validation", seed=42, image_size=(SIZE, SIZE), batch_size=32)
leaf_classes = tr.class_names
norm = lambda x, y: (x / 255.0, y)
tr, va = tr.map(norm).prefetch(2), va.map(norm).prefetch(2)
base = tf.keras.applications.MobileNetV2(input_shape=(SIZE, SIZE, 3), include_top=False, weights="imagenet"); base.trainable = False
inp = tf.keras.layers.Input((SIZE, SIZE, 3))
x = tf.keras.layers.RandomFlip("horizontal")(inp)
x = tf.keras.layers.Rescaling(2.0, offset=-1.0)(x)
x = base(x, training=False); x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x); x = tf.keras.layers.Dropout(0.3)(x)
m = tf.keras.Model(inp, tf.keras.layers.Dense(len(leaf_classes), activation="softmax")(x))
m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(tr, validation_data=va, epochs=3)
leaf_acc = round(float(m.evaluate(va, verbose=0)[1]), 4)
os.makedirs("out", exist_ok=True); m.save("out/leaf_cnn.keras")
print("Leaf validation accuracy:", leaf_acc)

## 2. Land-cover model (EuroSAT): convert the 13-band files to RGB images

In [ ]:
import glob, numpy as np, tifffile
from PIL import Image
src = kagglehub.dataset_download("apollo2506/eurosat-dataset")
base_dir = os.path.join(src, "EuroSATallBands")
classes = sorted(d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)))
dst = "/content/eurosat_rgb"
for c in classes:
    os.makedirs(f"{dst}/{c}", exist_ok=True)
    for f in glob.glob(f"{base_dir}/{c}/*.tif"):
        a = tifffile.imread(f).astype("float32")
        if a.shape[0] == 13: a = a.transpose(1, 2, 0)
        rgb = np.clip(a[:, :, [3, 2, 1]] / 3000.0, 0, 1) * 255          # B04, B03, B02 = red, green, blue
        Image.fromarray(rgb.astype("uint8")).save(f"{dst}/{c}/{os.path.basename(f)[:-4]}.png")
    print(c)

## 3. Train the land-cover model

In [ ]:
data_dir = "/content/eurosat_rgb"
tr = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="training", seed=42, image_size=(SIZE, SIZE), batch_size=32)
va = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset="validation", seed=42, image_size=(SIZE, SIZE), batch_size=32)
land_classes = tr.class_names
tr, va = tr.map(norm).prefetch(2), va.map(norm).prefetch(2)
base = tf.keras.applications.MobileNetV2(input_shape=(SIZE, SIZE, 3), include_top=False, weights="imagenet"); base.trainable = False
inp = tf.keras.layers.Input((SIZE, SIZE, 3))
x = tf.keras.layers.RandomFlip("horizontal")(inp)
x = tf.keras.layers.Rescaling(2.0, offset=-1.0)(x)
x = base(x, training=False); x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x); x = tf.keras.layers.Dropout(0.3)(x)
m = tf.keras.Model(inp, tf.keras.layers.Dense(len(land_classes), activation="softmax")(x))
m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(tr, validation_data=va, epochs=5)
land_acc = round(float(m.evaluate(va, verbose=0)[1]), 4)
m.save("out/landcover_cnn.keras")
print("Land-cover validation accuracy:", land_acc)

## 4. Save class lists and accuracies, then download

In [ ]:
import json
json.dump({"leaf_classes": leaf_classes, "leaf_val_accuracy": leaf_acc,
           "landcover_classes": land_classes, "landcover_val_accuracy": land_acc}, open("out/image_meta.json", "w"))
!cd out && zip -q ../agrisense_image_models.zip *
from google.colab import files; files.download("agrisense_image_models.zip")